In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from utils_firerisk import SEASON

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

DATA = Path("../data")
raw = pd.read_csv(DATA / "processed" / "panel_dev.csv", parse_dates=["date"])
print("Kích thước:", raw.shape, "| bộ nhớ: %.0f MB" % (raw.memory_usage(deep=True).sum() / 1e6))
print("Khoảng ngày:", raw["date"].min().date(), "->", raw["date"].max().date(), "| số tỉnh:", raw["prov"].nunique())
raw.dtypes.value_counts()

Kích thước: (273209, 78) | bộ nhớ: 172 MB
Khoảng ngày: 2012-01-20 -> 2024-03-31 | số tỉnh: 63


float64           56
int64             18
str                2
datetime64[us]     1
bool               1
Name: count, dtype: int64

In [2]:
dup_key = int(raw.duplicated(["prov", "date"]).sum())
dup_row = int(raw.duplicated().sum())
span = raw.groupby("prov")["date"].agg(["min", "max", "count"])
span["gap_days"] = (span["max"] - span["min"]).dt.days + 1 - span["count"]
print("Trùng theo (tỉnh, ngày):", dup_key, "| trùng toàn dòng:", dup_row)
print("Tỉnh có lỗ hổng ngày:", int((span["gap_days"] > 0).sum()), "| số dòng mỗi tỉnh min/max:", span["count"].min(), span["count"].max())

nulls = raw.isna().sum()
nulls = nulls[nulls > 0].to_frame("n_null")
nulls["pct"] = (nulls["n_null"] / len(raw) * 100).round(2)
nulls

Trùng theo (tỉnh, ngày): 0 | trùng toàn dòng: 0
Tỉnh có lỗ hổng ngày: 0 | số dòng mỗi tỉnh min/max: 4303 4455


,n_null,pct
wind13,273209,100.00
season,135330,49.53


Hai cột có null. wind13 rỗng 100% vì biến gió đã bị bỏ khỏi lời gọi API để giảm hạn mức, nên xóa cột. season rỗng 49,5% là null có chủ đích: đó là các ngày ngoài mùa cháy của vùng, không điền mà dùng cờ in_season.

In [3]:
assert raw["season"].isna().equals(raw["in_season"].eq(0))
ranges = raw.select_dtypes("number").agg(["min", "max"]).T
ranges.loc[["T13", "Td13", "RH13", "D13_hpa", "dd13_c", "rain_cal", "rain_30d"]].round(2)

,min,max
T13,2.30,40.90
Td13,-6.40,29.90
RH13,15.49,100.00
D13_hpa,0.00,51.55
dd13_c,0.00,27.20
rain_cal,0.00,315.80
rain_30d,0.00,1818.00


In [4]:
checks = {
    "RH13 trong [0,100]": bool(raw["RH13"].between(0, 100.0001).all()),
    "mưa >= 0": bool((raw["rain_cal"] >= 0).all()),
    "T13 >= Td13": bool((raw["dd13_c"] >= -1e-9).all()),
    "n_hot_forest <= n_hot_all": bool((raw["n_hot_forest"] <= raw["n_hot_all"]).all()),
    "fire_any_forest == (n_hot_forest > 0)": bool((raw["fire_any_forest"] == (raw["n_hot_forest"] > 0).astype(int)).all()),
    "cột tĩnh không đổi trong tỉnh": bool((raw.groupby("prov")[["area_km2", "forest_share", "rep_elev_m"]].nunique() == 1).all().all()),
    "không có chỉ số P null": int(raw.filter(like="P_").isna().sum().sum()) == 0,
    "sealed toàn False trong file dev": bool((~raw["sealed"]).all()),
}
pd.Series(checks, name="đạt")

RH13 trong [0,100]                       True
mưa >= 0                                 True
T13 >= Td13                              True
n_hot_forest <= n_hot_all                True
fire_any_forest == (n_hot_forest > 0)    True
cột tĩnh không đổi trong tỉnh            True
không có chỉ số P null                   True
sealed toàn False trong file dev         True
Name: đạt, dtype: bool

In [5]:
df = raw.drop(columns=["wind13", "sealed"]).copy()

for column in ["prov", "region"]:
    df[column] = df[column].astype("category")

int_columns = [
    "wx_missing", "fire_any_forest", "large_event_active", "is_weekend", "tet_win3", "giotuo_win1",
    "thanh_minh_win1", "holiday_solar", "in_season", "dow", "month",
]
df[int_columns] = df[int_columns].astype("int8")

float_columns = [c for c in df.select_dtypes("float64").columns if c != "season"]
df[float_columns] = df[float_columns].astype("float32")

print("Bộ nhớ sau khi ép kiểu: %.0f MB (trước: %.0f MB)" % (df.memory_usage(deep=True).sum() / 1e6, raw.memory_usage(deep=True).sum() / 1e6))
print("Số cột còn lại:", df.shape[1])

Bộ nhớ sau khi ép kiểu: 82 MB (trước: 172 MB)
Số cột còn lại: 76


Bản ghi điểm cháy chuẩn của VIIRS bắt đầu từ 2012-01-20, nên mùa nào bắt đầu trước ngày đó bị thiếu một phần dữ liệu cháy. Các dòng này được đánh dấu partial_season và không dùng cho mô hình, nhưng vẫn giữ để vẽ EDA. Đồng bằng sông Hồng và sông Cửu Long chủ yếu cháy đất nông nghiệp và đô thị nên chỉ làm nhóm đối chứng. Tỉnh đủ điều kiện phải có ít nhất 150 ngày cháy rừng trong mùa và ít nhất 6 mùa có từ 5 ngày cháy trở lên; ngưỡng này chốt trước khi phân tích (docs/prereg_v1.md).

In [6]:
FIRST_FIRE_DAY = pd.Timestamp("2012-01-20")
FOREST_REGIONS = ["TB", "DB", "BTB", "NTB", "TN", "DNB"]
MIN_FIRE_DAYS, MIN_SEASONS, MIN_DAYS_PER_SEASON = 150, 6, 5

season_first_month = df["region"].astype(str).map({k: v[0] for k, v in SEASON.items()})
season_start = pd.to_datetime(dict(year=df["season"].fillna(1900).astype(int), month=season_first_month, day=1))
df["partial_season"] = ((df["in_season"] == 1) & (season_start < FIRST_FIRE_DAY)).astype("int8")
print("Số dòng trong mùa:", int(df["in_season"].sum()), "| trong đó mùa không đầy đủ:", int(df["partial_season"].sum()))

in_full_season = df[(df["in_season"] == 1) & (df["partial_season"] == 0)]


def is_eligible(group):
    fire_days = group[group["fire_any_forest"] == 1]
    seasons_with_fire = (fire_days.groupby("season", observed=True).size() >= MIN_DAYS_PER_SEASON).sum()
    return len(fire_days) >= MIN_FIRE_DAYS and seasons_with_fire >= MIN_SEASONS


eligible_flags = in_full_season.groupby("prov", observed=True).apply(is_eligible)
eligible_provs = sorted(eligible_flags[eligible_flags].index.astype(str))

df["eligible_prov"] = df["prov"].astype(str).isin(eligible_provs).astype("int8")
df["use_for_model"] = (
    (df["in_season"] == 1) & (df["partial_season"] == 0)
    & df["region"].astype(str).isin(FOREST_REGIONS) & (df["eligible_prov"] == 1)
).astype("int8")

by_region = pd.DataFrame({
    "tỉnh": df.groupby("region", observed=True)["prov"].nunique(),
    "đủ điều kiện": df[df["eligible_prov"] == 1].groupby("region", observed=True)["prov"].nunique(),
}).fillna(0).astype(int)
print("Tổng số tỉnh đủ điều kiện:", len(eligible_provs), "| dòng dùng cho mô hình:", int(df["use_for_model"].sum()))
by_region.T

Số dòng trong mùa: 137879 | trong đó mùa không đầy đủ: 7542


Tổng số tỉnh đủ điều kiện: 46 | dòng dùng cho mô hình: 84355


region,BTB,DB,DBSCL,DBSH,DNB,NTB,TB,TN
tỉnh,6,11,13,10,6,8,4,5
đủ điều kiện,6,11,5,1,6,8,4,5


Giá trị cực đoan của mưa và nhiệt là hiện tượng thật (bão, đợt nắng nóng) chứ không phải lỗi đo, nên không xóa. Chỉ ghi nhận và dùng log1p cho các biến lệch phải.

In [7]:
def beyond_3iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    k = 3 * (q3 - q1)
    return ((s < q1 - k) | (s > q3 + k)).mean()


weather_columns = ["T13", "D13_hpa", "rain_cal", "rain_30d", "dd13_c"]
outliers = pd.DataFrame({
    "% vượt 3×IQR": {c: round(100 * beyond_3iqr(df[c]), 2) for c in weather_columns},
    "p99": {c: round(float(df[c].quantile(0.99)), 2) for c in weather_columns},
    "max": {c: round(float(df[c].max()), 2) for c in weather_columns},
})
print("Mưa ngày > 150 mm:", int((df["rain_cal"] > 150).sum()), "ngày")
outliers

Mưa ngày > 150 mm: 98 ngày


,% vượt 3×IQR,p99,max
T13,0.06,36.80,40.90
D13_hpa,0.01,33.69,51.55
rain_cal,2.70,44.70,315.80
rain_30d,0.15,571.00,1818.00
dd13_c,0.03,16.40,27.20


26 biến thể chỉ số Nesterov chỉ khác nhau ở ngưỡng mưa đặt lại, hệ số K và cách tính độ hụt ẩm nên rất giống nhau; đưa cả 26 vào mô hình tuyến tính sẽ gây đa cộng tuyến. Sau khi xem tương quan hạng Spearman, giữ biến chính cùng 4 biến thể khác ngưỡng hoặc cách tính; 21 biến còn lại vẫn nằm trong panel_dev.

In [8]:
p_columns = [c for c in df.columns if c.startswith("P_")]
sample = df.loc[df["use_for_model"] == 1, p_columns].sample(40000, random_state=0)
corr = sample.corr(method="spearman")
pairs = corr.values[np.triu_indices(len(p_columns), 1)]
print("Số biến thể:", len(p_columns), "| tương quan hạng giữa các cặp: min %.3f, median %.3f" % (pairs.min(), np.median(pairs)))
corr["P_linear5_Ddew"].sort_values().head(5).round(3)

Số biến thể: 26 | tương quan hạng giữa các cặp: min 0.764, median 0.908


P_binary10_Dsat    0.764
P_binary10_Ddew    0.779
P_linear10_Dsat    0.786
P_linear10_Ddew    0.796
P_binary9_Dsat     0.797
Name: P_linear5_Ddew, dtype: float64

In [9]:
KEEP_P = ["P_linear5_Ddew", "P_linear7_Dsat", "P_binary5_Dsat", "P_binary10_Ddew", "P_alt19_linear7_Dsat"]
df = df.drop(columns=[c for c in p_columns if c not in KEEP_P])
df["P_primary"] = df["P_linear5_Ddew"]
df["logP"] = np.log1p(df["P_primary"]).astype("float32")
df["log_rain_30d"] = np.log1p(df["rain_30d"]).astype("float32")
df["log_rain_cal"] = np.log1p(df["rain_cal"]).astype("float32")
df.filter(like="P_").describe().T[["mean", "50%", "max"]].round(1)

,mean,50%,max
P_linear5_Ddew,2265.7,308.6,56931.3
P_binary5_Dsat,4489.6,798.8,128232.2
P_linear7_Dsat,5226.0,944.3,145277.5
P_binary10_Ddew,4003.0,1080.7,72833.4
P_alt19_linear7_Dsat,5262.6,965.3,136004.2
P_primary,2265.7,308.6,56931.3


In [10]:
df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25).astype("float32")
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25).astype("float32")
first_month = df["region"].astype(str).map({k: v[0] for k, v in SEASON.items()})
df["month_in_season"] = (((df["month"] - first_month) % 12) + 1).where(df["in_season"] == 1).astype("float32")
df["fire_rate_clim_prov_month"] = df.groupby(["prov", "month"], observed=True)["fire_any_forest"].transform("mean").astype("float32")
df[["doy_sin", "doy_cos", "month_in_season", "logP"]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
doy_sin,273209.0,0.00,0.71,-1.0,-0.71,0.01,0.72,1.00
doy_cos,273209.0,-0.01,0.70,-1.0,-0.71,-0.02,0.69,1.00
month_in_season,137879.0,3.68,1.84,1.0,2.00,4.00,5.00,8.00
logP,273209.0,4.63,3.51,0.0,0.00,5.74,7.47,10.95


fire_rate_clim_prov_month chỉ dùng cho EDA, không đưa vào mô hình vì rò rỉ nhãn. Dữ liệu chuỗi thời gian không chia ngẫu nhiên (các ngày kề nhau rò rỉ thông tin) mà chia theo mùa cháy: huấn luyện đến 2018, kiểm định 2019 đến 2020, kiểm tra từ 2021. Phần bài báo còn có tập niêm phong riêng gồm hai mùa cuối.

In [11]:
TRAIN_LAST_SEASON, VALID_LAST_SEASON = 2018, 2020
season = df["season"]
df["split"] = np.select(
    [df["use_for_model"] == 0, season <= TRAIN_LAST_SEASON, season <= VALID_LAST_SEASON],
    ["khong_dung", "train", "valid"],
    default="test",
)
model_rows = df[df["use_for_model"] == 1]
model_rows.groupby("split", observed=True).agg(
    ngay=("fire_any_forest", "size"),
    ty_le_chay=("fire_any_forest", "mean"),
    ty_le_su_kien_lon=("large_event_active", "mean"),
).round(3)

,ngay,ty_le_chay,ty_le_su_kien_lon
split,,,
test,18020,0.406,0.174
train,51143,0.419,0.218
valid,15192,0.445,0.195


Trong mẫu mô hình khoảng 40 đến 45% số ngày có cháy (toàn bảng chỉ 21%), nên nhãn không quá lệch. Sự kiện lớn (khoảng 20%) thì lệch hơn, vì vậy đánh giá bằng PR-AUC và Brier skill thay cho accuracy.

In [12]:
model_rows.groupby(["region", "split"], observed=True)["fire_any_forest"].agg(["size", "mean"]).round(3).unstack("split")

size                mean              
split   test  train valid   test  train  valid
region                                        
BTB     2754   6426  1836  0.418  0.426  0.478
DB      3982  13948  3993  0.285  0.228  0.242
DNB     2184   7650  2190  0.325  0.365  0.383
NTB     5832  11672  3896  0.442  0.474  0.524
TB      1448   5072  1452  0.459  0.477  0.496
TN      1820   6375  1825  0.595  0.747  0.722

In [13]:
CLEAN = DATA / "clean"
CLEAN.mkdir(exist_ok=True)

df.to_csv(CLEAN / "panel_clean.csv", index=False, float_format="%.5g")
pd.DataFrame({"prov": eligible_provs}).to_csv(CLEAN / "eligible_provinces.csv", index=False)

column_groups = {
    "khóa": ["date", "prov", "region"],
    "thời tiết 13h": ["T13", "Td13", "RH13", "D13_hpa", "dd13_c"],
    "mưa": ["rain_cal", "rain_19_19", "rain_3d", "rain_7d", "rain_14d", "rain_30d", "dsr_1", "dsr_5", "dsr_10", "log_rain_cal", "log_rain_30d"],
    "Nesterov": [c for c in df.columns if c.startswith("P_") or c == "logP"],
    "độ lệch/trung bình trượt": ["T13_roll7", "D13_roll7", "T13_anom", "D13_anom"],
    "điểm cháy (kết quả)": ["n_hot_all", "n_hot_forest", "fire_any_forest", "event_start", "large_event_active"],
    "lịch": [
        "dow", "is_weekend", "month", "doy", "year", "days_from_tet", "tet_win3", "days_from_giotuo",
        "giotuo_win1", "thanh_minh_win1", "holiday_solar", "doy_sin", "doy_cos", "month_in_season",
    ],
    "ENSO": ["oni", "oni_lag3"],
    "mùa/mẫu": ["season", "in_season", "partial_season", "eligible_prov", "use_for_model", "split"],
    "tĩnh theo tỉnh": ["area_km2", "forest_share", "crop_share", "grass_share", "built_share", "rep_elev_m", "rep_lat", "rep_lon"],
    "khác": ["wx_missing", "fire_rate_clim_prov_month"],
}
group_of = {column: group for group, columns in column_groups.items() for column in columns}

dictionary = pd.DataFrame({
    "cot": df.columns,
    "nhom": [group_of.get(c, "?") for c in df.columns],
    "kieu": [str(t) for t in df.dtypes],
    "pct_null": [round(100 * df[c].isna().mean(), 2) for c in df.columns],
})
assert (dictionary["nhom"] != "?").all(), dictionary[dictionary["nhom"] == "?"]
dictionary.to_csv(CLEAN / "data_dictionary.csv", index=False)

print("Đã lưu panel_clean.csv:", df.shape, "| null còn lại:", dictionary[dictionary["pct_null"] > 0].set_index("cot")["pct_null"].to_dict())
dictionary.groupby("nhom").size().to_frame("số cột")

Đã lưu panel_clean.csv: (273209, 67) | null còn lại: {'season': 49.53, 'month_in_season': 49.53}


,số cột
nhom,
ENSO,2
Nesterov,7
khác,2
khóa,3
lịch,14
mùa/mẫu,6
mưa,11
thời tiết 13h,5
tĩnh theo tỉnh,8
